## Question 2
### Import Library & Configuration

In [3]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM


### Preprocessing functions

In [4]:
CSV_PATH = "test.csv"  

def clean_basic(text: str) -> str:
    # get rid of whitespaces
    return str(text).strip()

def remove_punc_lower(text: str) -> str:
    text = str(text).lower()
    # keep only letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_into_sentences(text: str):
    # Simple sentence splitting by . ! ?
    sents = re.split(r"[.!?]", text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents

def build_prefix_target_pairs(sent_list, lowercase=True):
    #Input: list of sentences
    #Output: (prefix, target) pairs, where the last word of each sentence is the target
    pairs = []
    for s in sent_list:
        if lowercase:
            s = s.lower()
        tokens = s.split()
        if len(tokens) < 2:
            continue
        prefix = " ".join(tokens[:-1])
        target = tokens[-1]
        pairs.append((prefix, target))
    return pairs


### Load data & construct two sentence versions (with / without punctuation)

In [5]:
# read AG News test.csv third column
df = pd.read_csv(CSV_PATH, header=None)
raw_texts = df.iloc[:, 2].astype(str).tolist()

cleaned_basic = [clean_basic(t) for t in raw_texts]

# For each doc, select the longest sentence (with punctuation)
sentences_with_punc = []
for txt in cleaned_basic:
    sents = split_into_sentences(txt)
    if not sents:
        continue
    longest = max(sents, key=lambda s: len(s.split()))
    sentences_with_punc.append(longest)

print("Num sentences with punctuation:", len(sentences_with_punc))

# Build lowercase versions without punctuation
sentences_no_punc = [remove_punc_lower(s) for s in sentences_with_punc]

# Build (prefix, target) pairs
pairs_no_punc = build_prefix_target_pairs(sentences_no_punc, lowercase=True)
pairs_with_punc = build_prefix_target_pairs(sentences_with_punc, lowercase=True)

print("Pairs (no punctuation):", len(pairs_no_punc))
print("Pairs (with punctuation):", len(pairs_with_punc))

# # To speed up, limit the number of pairs
# MAX_PAIRS = 1000
# pairs_no_punc = pairs_no_punc[:MAX_PAIRS]
# pairs_with_punc = pairs_with_punc[:MAX_PAIRS]


Num sentences with punctuation: 7600
Pairs (no punctuation): 7600
Pairs (with punctuation): 7600


### BERT fill-mask settings + prediction function

In [8]:
# BERT: masked language model
bert_fill = pipeline('fill-mask', model='bert-base-uncased')

def predict_next_word_bert(prefix: str) -> str:
    #Input the first n-1 words, with the last position as [MASK]
    prompt = prefix.strip() + " [MASK]"
    preds = bert_fill(prompt)
    # Take top-1 prediction
    return preds[0]['token_str'].strip()


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


### GPT-2 Configuration + Prediction Function

In [9]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")

def predict_next_word_gpt2(prefix: str) -> str:
    inputs = gpt2_tokenizer(prefix, return_tensors="pt")
    with torch.no_grad():
        logits = gpt2_model(**inputs).logits[:, -1, :]
    pred_id = torch.argmax(logits, dim=-1).item()
    pred_word = gpt2_tokenizer.decode(pred_id).strip()
    return pred_word


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### General function for calculating accuracy

In [10]:
def compute_accuracy(pairs, model_name="bert"):
    correct = 0
    total = len(pairs)
    for prefix, target in tqdm(pairs, desc=f"Evaluating {model_name}"):
        try:
            if model_name == "bert":
                pred = predict_next_word_bert(prefix)
            elif model_name == "gpt2":
                pred = predict_next_word_gpt2(prefix)
            else:
                raise ValueError("Unknown model name")
        except Exception as e:
            # skip one sample on error
            total -= 1
            continue
        if pred.lower() == target.lower():
            correct += 1
    if total == 0:
        return 0.0
    return correct / total


### Evaluating BERT / GPT-2 under “without punctuation” conditions

In [11]:
acc_bert_no_punc = compute_accuracy(pairs_no_punc, model_name="bert")
acc_gpt2_no_punc = compute_accuracy(pairs_no_punc, model_name="gpt2")

print("BERT accuracy (no punctuation):", acc_bert_no_punc)
print("GPT-2 accuracy (no punctuation):", acc_gpt2_no_punc)


Evaluating gpt2: 100%|██████████| 7600/7600 [03:32<00:00, 35.82it/s]

BERT accuracy (no punctuation): 0.004605263157894736
GPT-2 accuracy (no punctuation): 0.23157894736842105


### Evaluating BERT / GPT-2 under "with punctuation” conditions

In [12]:
acc_bert_with_punc = compute_accuracy(pairs_with_punc, model_name="bert")
acc_gpt2_with_punc = compute_accuracy(pairs_with_punc, model_name="gpt2")

print("BERT accuracy (with punctuation):", acc_bert_with_punc)
print("GPT-2 accuracy (with punctuation):", acc_gpt2_with_punc)


Evaluating gpt2: 100%|██████████| 7600/7600 [03:51<00:00, 32.81it/s]

BERT accuracy (with punctuation): 0.005263157894736842
GPT-2 accuracy (with punctuation): 0.24552631578947368
